# 06 Boolean Targeting Walkthrough

This notebook shows how the demo can be extended to support richer campaign targeting without turning Redis into a full query engine.

The practical split is:
- use Redis Sets for a high-recall candidate domain,
- use app-side logic for exact evaluation of more complicated boolean rules,
- rerank only the surviving campaigns.

That means we can support structured filters such as `all_of`, `any_of`, and `none_of` directly in the candidate-generation layer, while still leaving room for exact expression checks such as XOR or nested branch-specific negation.

In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import sys

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise RuntimeError('Could not locate repo root')


REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from app.models import UserProfile

# Hand-built demo MAID. The synthetic generator does not produce the
# exact segment combination we want to walk through here, so the example
# is constructed inline so each case below is reproducible.
sample_user = UserProfile(
    user_id='maid_demo',
    geo='US',
    state='NY',
    postal_code='10001',
    device='iOS',
    device_type='mobile',
    age_bucket='25-34',
    card_tier='Gold',
    spend_tier='medium',
    interests={
        'camping': 0.81, 'gaming': 0.74, 'travel': 0.69,
        'luxury': 0.71, 'home_improvement': 0.62, 'pet_care': 0.32,
        'family': 0.18, 'streaming': 0.46,
    },
    segments=['camping_high', 'gaming_high', 'travel_high', 'luxury_high', 'home_improvement_high'],
    impression_count=12,
    frequency_history={},
)
sample_user

UserProfile(user_id='maid_demo', geo='US', device='iOS', age_bucket='25-34', interests={'camping': 0.81, 'gaming': 0.74, 'travel': 0.69, 'luxury': 0.71, 'home_improvement': 0.62, 'pet_care': 0.32, 'family': 0.18, 'streaming': 0.46}, segments=['camping_high', 'gaming_high', 'travel_high', 'luxury_high', 'home_improvement_high'], identity_tokens=[], state='NY', postal_code='10001', device_type='mobile', card_tier='Gold', spend_tier='medium', frequency_history={}, impression_count=12)

## Example Campaign Targeting Shapes

These example campaigns use the same segment vocabulary as the synthetic demo, but they carry richer targeting metadata than the current API implementation.

Two patterns are shown:
- a structured form that Redis can mostly evaluate with set algebra,
- a more complex `exact_expression` that should be evaluated in the app after retrieval.

In [2]:
campaigns = [
    {
        'campaign_id': 'bool_001',
        'geo': ['US'],
        'device': ['iOS', 'Android'],
        'all_of': ['camping_high'],
        'any_of': ['travel_high', 'family_high'],
        'none_of': ['gaming_high'],
        'exact_expression': None,
        'notes': 'Filtered by app-side none_of (user has gaming_high)',
    },
    {
        'campaign_id': 'bool_002',
        'geo': ['US'],
        'device': ['iOS'],
        'all_of': ['home_improvement_high'],
        'any_of': ['pet_care_high', 'family_high'],
        'none_of': ['luxury_high'],
        'exact_expression': None,
        'notes': 'Filtered by app-side any_of (user has neither pet_care_high nor family_high)',
    },
    {
        'campaign_id': 'bool_003',
        'geo': ['US'],
        'device': ['iOS'],
        'all_of': [],
        'any_of': ['camping_high', 'gaming_high', 'travel_high'],
        'none_of': [],
        'exact_expression': {
            'or': [
                {'xor': ['camping_high', 'gaming_high']},
                {'and': ['travel_high', {'not': 'luxury_high'}]},
            ]
        },
        'notes': 'Filtered by app-side XOR (user has both camping_high and gaming_high)',
    },
    {
        'campaign_id': 'bool_004',
        'geo': ['US'],
        'device': ['iOS'],
        'all_of': ['travel_high'],
        'any_of': ['home_improvement_high', 'family_high'],
        'none_of': ['pet_care_high'],
        'exact_expression': None,
        'notes': 'Survives coarse and exact stages, eligible for rerank',
    },
]

pd.DataFrame(campaigns)[['campaign_id', 'all_of', 'any_of', 'none_of', 'notes']]

  campaign_id                   all_of                                    any_of          none_of                                                                         notes
0    bool_001           [camping_high]                [travel_high, family_high]    [gaming_high]                           Filtered by app-side none_of (user has gaming_high)
1    bool_002  [home_improvement_high]              [pet_care_high, family_high]    [luxury_high]  Filtered by app-side any_of (user has neither pet_care_high nor family_high)
2    bool_003                       []  [camping_high, gaming_high, travel_high]               []         Filtered by app-side XOR (user has both camping_high and gaming_high)
3    bool_004            [travel_high]      [home_improvement_high, family_high]  [pet_care_high]                         Survives coarse and exact stages, eligible for rerank

## Build Inverted Indexes For The Structured Filters

For the `all_of` / `any_of` / `none_of` fields, the indexing strategy is the same as the current demo:
- each targeting bucket gets an `idx:segment:<bucket>` set,
- each campaign ID is inserted into the sets for the buckets it references.

The difference is that `any_of` and `none_of` are now first-class parts of the targeting schema.

In [3]:
indexes = defaultdict(set)
for campaign in campaigns:
    for geo in campaign['geo']:
        indexes[f'idx:geo:{geo}'].add(campaign['campaign_id'])
    for device in campaign['device']:
        indexes[f'idx:device:{device}'].add(campaign['campaign_id'])
    for segment in set(campaign['all_of']) | set(campaign['any_of']) | set(campaign['none_of']):
        indexes[f'idx:segment:{segment}'].add(campaign['campaign_id'])

pd.DataFrame(
    [
        {'key': key, 'members': sorted(values)}
        for key, values in sorted(indexes.items())
    ]
)

                                 key                                   members
0                 idx:device:Android                                [bool_001]
1                     idx:device:iOS  [bool_001, bool_002, bool_003, bool_004]
2                         idx:geo:US  [bool_001, bool_002, bool_003, bool_004]
3           idx:segment:camping_high                      [bool_001, bool_003]
4            idx:segment:family_high            [bool_001, bool_002, bool_004]
5            idx:segment:gaming_high                      [bool_001, bool_003]
6  idx:segment:home_improvement_high                      [bool_002, bool_004]
7            idx:segment:luxury_high                                [bool_002]
8          idx:segment:pet_care_high                      [bool_002, bool_004]
9            idx:segment:travel_high            [bool_001, bool_003, bool_004]

## Coarse Candidate Domain Per User

At bid time we have a user, and we want a small set of campaigns that *might* match.
The Redis plan in this prototype is:

- intersect `idx:geo:<user.geo>` with `idx:device:<user.device>` to filter on hard targeting,
- union all of the user's `idx:segment:<segment>` sets to capture every campaign that mentions any of the user's segments,
- intersect those two so the coarse domain is small but high-recall.

This deliberately over-approximates: a campaign whose `all_of` mentions one of the user's segments is in the coarse domain even if the campaign also requires another segment the user does not have. The exact stage below catches that.

In [4]:
def coarse_candidate_domain(user: UserProfile, indexes: dict[str, set[str]]) -> tuple[set[str], list[str]]:
    """Return the coarse candidate set for the user and the equivalent Redis pipeline."""
    base_keys = [f'idx:geo:{user.geo}', f'idx:device:{user.device}']
    base = set.intersection(*(indexes.get(key, set()) for key in base_keys))

    commands = [f'SINTERSTORE tmp:base ' + ' '.join(base_keys)]
    if user.segments:
        segment_keys = [f'idx:segment:{segment}' for segment in user.segments]
        segment_pool = set().union(*(indexes.get(key, set()) for key in segment_keys))
        commands.append('SUNIONSTORE tmp:segments ' + ' '.join(segment_keys))
        commands.append('SINTERSTORE tmp:domain tmp:base tmp:segments')
        commands.append('SMEMBERS tmp:domain')
        return base & segment_pool, commands
    commands.append('SMEMBERS tmp:base')
    return base, commands


domain, command_plan = coarse_candidate_domain(sample_user, indexes)
print('coarse candidate domain:', sorted(domain))
print()
print('Redis pipeline:')
for command in command_plan:
    print(' ', command)

coarse candidate domain: ['bool_001', 'bool_002', 'bool_003', 'bool_004']

Redis pipeline:
  SINTERSTORE tmp:base idx:geo:US idx:device:iOS
  SUNIONSTORE tmp:segments idx:segment:camping_high idx:segment:gaming_high idx:segment:travel_high idx:segment:luxury_high idx:segment:home_improvement_high
  SINTERSTORE tmp:domain tmp:base tmp:segments
  SMEMBERS tmp:domain


## Exact App-Side Evaluation For More Complicated Rules

Structured fields are enough for many targeting use cases, but they are not enough for everything.

Example: `(camping_high XOR gaming_high) OR (travel_high AND NOT luxury_high)`

A good production pattern is:
- use Redis to over-approximate the candidate domain with high recall,
- run the exact expression in the app on the small candidate set,
- only rerank campaigns that pass the exact test.

The exact evaluator below operates over the user's segment set.

In [5]:
def evaluate_expression(node: object, user_segments: set[str]) -> bool:
    if isinstance(node, str):
        return node in user_segments
    if not isinstance(node, dict):
        raise TypeError(f'Unsupported node: {node!r}')
    if 'and' in node:
        return all(evaluate_expression(child, user_segments) for child in node['and'])
    if 'or' in node:
        return any(evaluate_expression(child, user_segments) for child in node['or'])
    if 'not' in node:
        return not evaluate_expression(node['not'], user_segments)
    if 'xor' in node:
        children = [evaluate_expression(child, user_segments) for child in node['xor']]
        return sum(children) == 1
    raise ValueError(f'Unsupported expression node: {node}')


def passes_exact_targeting(user: UserProfile, campaign: dict) -> bool:
    user_segments = set(user.segments)
    if campaign['all_of'] and not set(campaign['all_of']).issubset(user_segments):
        return False
    if campaign['any_of'] and not user_segments.intersection(campaign['any_of']):
        return False
    if campaign['none_of'] and user_segments.intersection(campaign['none_of']):
        return False
    if campaign['exact_expression'] is not None:
        return evaluate_expression(campaign['exact_expression'], user_segments)
    return True


user_segments = set(sample_user.segments)
print('user segments:', sorted(user_segments))
print("bool_003 exact_expression evaluates to:", evaluate_expression(campaigns[2]['exact_expression'], user_segments))

user segments: ['camping_high', 'gaming_high', 'home_improvement_high', 'luxury_high', 'travel_high']
bool_003 exact_expression evaluates to: False


## Putting The Two Stages Together

For the more complicated campaign, the coarse domain is intentionally broader than the final exact rule.
That is the whole point: Redis narrows the search space quickly, then the app restores exactness.

In [6]:
domain, _ = coarse_candidate_domain(sample_user, indexes)

combined_rows = []
for campaign in campaigns:
    in_domain = campaign['campaign_id'] in domain
    exact_ok = passes_exact_targeting(sample_user, campaign)
    combined_rows.append(
        {
            'campaign_id': campaign['campaign_id'],
            'in_coarse_domain': in_domain,
            'passes_exact_targeting': exact_ok,
            'eligible_for_rerank': in_domain and exact_ok,
            'reason': campaign['notes'],
        }
    )

pd.DataFrame(combined_rows)

  campaign_id  in_coarse_domain  passes_exact_targeting  eligible_for_rerank                                                                        reason
0    bool_001              True                   False                False                           Filtered by app-side none_of (user has gaming_high)
1    bool_002              True                   False                False  Filtered by app-side any_of (user has neither pet_care_high nor family_high)
2    bool_003              True                   False                False         Filtered by app-side XOR (user has both camping_high and gaming_high)
3    bool_004              True                    True                 True                         Survives coarse and exact stages, eligible for rerank

## Practical Takeaway

The set-based candidate domain plus the app-side exact evaluator covers the AND / OR / NOT and XOR rules in the example campaigns above without any secondary indexing engine.
The same pattern extends to richer expressions by:

- adding `all_of`, `any_of`, and `none_of` to the campaign schema,
- driving the coarse domain with `SINTERSTORE`, `SUNIONSTORE`, and `SDIFFSTORE` over the existing `idx:segment:*` sets,
- evaluating any nested boolean expression on the surviving candidate set,
- reranking only after the exact stage.

Boolean targeting on string segments has a practical limit: it does not express thresholds on continuous interest scores. The next notebook covers that case directly.